# Módulo 06 · Recuperación híbrida

Se combinan la búsqueda léxica (BM25) y la semántica (embeddings) mediante Reciprocal
Rank Fusion (RRF).

In [1]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR = ROOT / "data" / "novatech"

## Reciprocal Rank Fusion

Cada documento suma `peso / (60 + posición)` por cada ranking en el que aparece. Solo
se usan posiciones, por lo que no es necesario que las puntuaciones de ambos métodos
estén en la misma escala.

In [2]:
from fundamenta.retriever import reciprocal_rank_fusion

lexical_ranking = ["A", "B", "C"]
semantic_ranking = ["C", "A", "D"]
ids = {name: i for i, name in enumerate("ABCD")}
fused = reciprocal_rank_fusion(
    [[ids[n] for n in lexical_ranking], [ids[n] for n in semantic_ranking]]
)
for index, score in fused:
    print(f"{'ABCD'[index]}: {score:.5f}")

A: 0.03252
C: 0.03227
B: 0.01613
D: 0.01587


"A" y "C" aparecen en ambos rankings y quedan por delante de "B" y "D", que solo
aparecen en uno.

## Comparación de los tres modos

In [3]:
from fundamenta.chunking import chunk_documents
from fundamenta.embeddings import SentenceTransformerEmbedder
from fundamenta.loaders import load_directory
from fundamenta.retriever import Retriever

chunks = chunk_documents(load_directory(DATA_DIR))
retriever = Retriever(chunks, embedder=SentenceTransformerEmbedder(), mode="hybrid")

questions = [
    "¿Cuánto cuesta el plan Growth de NovaDesk?",
    "Me han robado el laptop, ¿qué tengo que hacer?",
    "¿Qué es NovaConnect?",
]
for question in questions:
    print(f"Consulta: {question}")
    for mode in ("lexical", "semantic", "hybrid"):
        retriever.mode = mode
        results = retriever.search(question, k=1)
        if not results:
            print(f"  {mode:9} -> (sin resultados)")
            continue
        top = results[0].document
        print(f"  {mode:9} -> {top.source} | {top.text[:60].replace(chr(10), ' ')}…")
    print()
retriever.mode = "hybrid"

Consulta: ¿Cuánto cuesta el plan Growth de NovaDesk?
  lexical   -> productos-y-precios.md | ### Planes de NovaDesk  | Plan        | Precio por agente y …
  semantic  -> productos-y-precios.md | ### Planes de NovaDesk  | Plan        | Precio por agente y …
  hybrid    -> productos-y-precios.md | ### Planes de NovaDesk  | Plan        | Precio por agente y …

Consulta: Me han robado el laptop, ¿qué tengo que hacer?
  lexical   -> (sin resultados)
  semantic  -> faq-soporte-it.txt | ¿Qué hago si pierdo el portátil o me lo roban? Avisa a IT de…
  hybrid    -> faq-soporte-it.txt | ¿Qué hago si pierdo el portátil o me lo roban? Avisa a IT de…

Consulta: ¿Qué es NovaConnect?
  lexical   -> faq-soporte-it.txt | ¿Cómo me conecto a la VPN? Instala el cliente NovaConnect de…
  semantic  -> faq-soporte-it.txt | ¿Cómo me conecto a la VPN? Instala el cliente NovaConnect de…


  hybrid    -> faq-soporte-it.txt | ¿Cómo me conecto a la VPN? Instala el cliente NovaConnect de…



## Filtros y diversidad

Los filtros por metadatos se aplican a ambos métodos antes de la fusión. La opción
`diversity` activa MMR (*Maximal Marginal Relevance*) para evitar resultados
redundantes.

In [4]:
for diversity in (None, 0.7):
    print(f"diversity={diversity}")
    for result in retriever.search("vacaciones y días libres", k=3, diversity=diversity):
        doc = result.document
        label = doc.metadata.get("section") or doc.text[:50].replace(chr(10), " ")
        print(f"  {result.rank}. {doc.source} | {label}")

diversity=None
  1. manual-del-empleado.md | Manual del empleado de NovaTech > Vacaciones
  2. beneficios-2026.pdf | 6. Día de voluntariado Cada persona tiene 1 día la
  3. manual-del-empleado.md | Manual del empleado de NovaTech > Teletrabajo
diversity=0.7
  1. manual-del-empleado.md | Manual del empleado de NovaTech > Vacaciones
  2. politica-de-gastos-y-viajes.md | Política de gastos y viajes > Dietas
  3. faq-soporte-it.txt | ¿Cada cuánto tengo que cambiar mi contraseña? Las 


El módulo 09 mide cuál de los tres modos funciona mejor sobre un conjunto de preguntas
de referencia.